In [3]:
#"""Neural-network adaptive PID control of a nonlinear thrust AMB.

#The model explicitly contains rotor mechanics, nonlinear magnetic force,
#voltage-limited amplifier dynamics, coil R-L dynamics, bias current, noisy
#position sensing, a second-order anti-alias filter, digital delay, actuator
#saturation, anti-windup, and independently configurable axial loads.

#Sign convention
#---------------
#The air gap ``x`` is positive away from the electromagnet. Gravity and a
#positive external thrust increase x; magnetic force decreases x:

#    m*x_ddot = m*g + F_load - c*x_dot - F_mag(x, i)
#    F_mag     = k_f*(i/x)^2 * saturation/fringing correction
#    L*i_dot   = v_coil - R*i
#
#The controller regulates the *absolute* air gap and commands total current
#around the equilibrium bias current I0.

#Run:
#    python Neural_Adaptive_PID_Thrust_AMB.py

#Only NumPy and Matplotlib are required.  The argument parser is Jupyter-safe.
#"""#
from __future__ import annotations

import argparse
import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Dict, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np


@dataclass
class PlantParameters:
    # Mechanical and electromagnetic parameters (SI units)
    mass: float = 0.42
    gravity: float = 9.81
    viscous_damping: float = 0.45
    nominal_gap: float = 0.014
    initial_gap: float = 0.0145
    gap_min: float = 0.006
    gap_max: float = 0.024
    hard_stop_min: float = 0.0035
    hard_stop_max: float = 0.030
    force_coefficient: float = 1.80e-4  # N*m^2/A^2
    current_bias: float = 0.0            # 0 => compute exact equilibrium value
    current_min: float = 0.0
    current_max: float = 5.0
    coil_resistance: float = 3.2         # ohm
    coil_inductance: float = 18e-3       # H

    # Power amplifier / inner current-loop parameters
    current_loop_gain: float = 18.0      # V/A
    supply_voltage: float = 48.0         # symmetric +/- limit [V]
    amplifier_gain: float = 1.0
    amplifier_efficiency: float = 0.97
    amplifier_time_constant: float = 1.5e-3

    # Weak effects deliberately omitted from the controller's Jacobian model
    current_saturation_shape: float = 0.045
    fringing_coefficient: float = 0.018
    contact_stiffness: float = 2.0e4
    contact_damping: float = 35.0

    def __post_init__(self) -> None:
        if self.current_bias <= 0.0:
            self.current_bias = np.sqrt(
                self.mass * self.gravity * self.nominal_gap**2
                / self.force_coefficient
            )


@dataclass
class SensorParameters:
    sensitivity: float = 800.0           # V/m
    bias_voltage: float = 0.012           # uncorrected sensor offset [V]
    noise_std_voltage: float = 8.0e-4
    cutoff_hz: float = 85.0
    sample_time: float = 1.0e-3


@dataclass
class ControllerParameters:
    sample_time: float = 1.0e-3
    computational_delay_samples: int = 1
    hidden_neurons: int = 10
    learning_rate: float = 8.0e-4
    gradient_clip: float = 3.0
    weight_decay: float = 2.0e-5
    adaptation_deadband: float = 20e-6
    normalized_error_scale: float = 1.0e-3
    jacobian_horizon: float = 0.030
    derivative_filter_tau: float = 5.0e-3
    integrator_limit: float = 0.020
    antiwindup_gain: float = 5.0
    # Strictly positive safe intervals for [Kp, Ki, Kd].
    gain_min: Tuple[float, float, float] = (80.0, 10.0, 0.50)
    gain_max: Tuple[float, float, float] = (700.0, 1000.0, 24.0)
    gain_initial: Tuple[float, float, float] = (380.0, 520.0, 6.0)


class ThrustAMBPlant:
    #"""Nonlinear four-state AMB plant integrated with classical RK4.

    #State = [gap, gap_velocity, coil_current, amplifier_output_voltage].
    #The amplifier demand is generated by a proportional inner current loop
    #with resistance feed-forward and supply-voltage saturation.
    #"""

    def __init__(self, p: PlantParameters):
        self.p = p
        self.state = np.array([p.initial_gap, 0.0, p.current_bias, p.coil_resistance*p.current_bias])
        self.last_voltage_demand = 0.0
        self.last_force = 0.0

    def magnetic_force(self, gap: float, current: float) -> float:
        p = self.p
        gap_eff = max(gap, p.hard_stop_min)
        current = float(np.clip(current, p.current_min, p.current_max))
        ideal = p.force_coefficient * (current / gap_eff) ** 2
        saturation = 1.0 - p.current_saturation_shape*np.tanh(0.9*(current-2.2))
        fringing = 1.0/(1.0 + p.fringing_coefficient*p.nominal_gap/gap_eff)
        return ideal*saturation*fringing

    def _derivative(self, state: np.ndarray, current_command: float, load: float) -> np.ndarray:
        p = self.p
        x, velocity, current, amplifier_voltage = state
        current_command = float(np.clip(current_command, p.current_min, p.current_max))
        voltage_demand = p.coil_resistance*current_command + p.current_loop_gain*(current_command-current)
        voltage_demand = float(np.clip(voltage_demand, -p.supply_voltage, p.supply_voltage))
        target_voltage = p.amplifier_efficiency*p.amplifier_gain*voltage_demand
        voltage_dot = (target_voltage-amplifier_voltage)/p.amplifier_time_constant
        current_dot = (amplifier_voltage-p.coil_resistance*current)/p.coil_inductance

        magnetic_force = self.magnetic_force(x, current)
        contact_force = 0.0
        if x < p.hard_stop_min:
            contact_force += p.contact_stiffness*(p.hard_stop_min-x)-p.contact_damping*velocity
        elif x > p.hard_stop_max:
            contact_force -= p.contact_stiffness*(x-p.hard_stop_max)-p.contact_damping*velocity
        acceleration = (
            p.mass*p.gravity + load - p.viscous_damping*velocity
            - magnetic_force + contact_force
        )/p.mass
        self.last_voltage_demand = voltage_demand
        return np.array([velocity, acceleration, current_dot, voltage_dot])

    def step(self, current_command: float, load: float, dt: float, substeps: int = 5) -> None:
        h = dt/substeps
        for _ in range(substeps):
            s = self.state
            k1 = self._derivative(s, current_command, load)
            k2 = self._derivative(s+0.5*h*k1, current_command, load)
            k3 = self._derivative(s+0.5*h*k2, current_command, load)
            k4 = self._derivative(s+h*k3, current_command, load)
            self.state = s+(h/6.0)*(k1+2*k2+2*k3+k4)
        self.state[2] = np.clip(self.state[2], self.p.current_min, self.p.current_max)
        self.last_force = self.magnetic_force(self.state[0], self.state[2])


class SensorFilter:
    #"""Position transducer plus a digital 2nd-order Butterworth low-pass.

   #The bilinear-transform coefficients are calculated directly, avoiding a
    #SciPy dependency. Sensor bias is intentionally left uncompensated so its
    #influence can be studied; set ``bias_voltage=0`` for ideal calibration.
    #"""

    def __init__(self, p: SensorParameters, initial_gap: float, seed: int = 12):
        self.p = p
        self.rng = np.random.default_rng(seed)
        k = np.tan(np.pi*p.cutoff_hz*p.sample_time)
        norm = 1.0/(1.0+np.sqrt(2.0)*k+k*k)
        self.b = np.array([k*k*norm, 2*k*k*norm, k*k*norm])
        self.a = np.array([1.0, 2*(k*k-1)*norm, (1-np.sqrt(2)*k+k*k)*norm])
        initial_voltage = p.sensitivity*initial_gap+p.bias_voltage
        self.x_hist = np.array([initial_voltage, initial_voltage])
        self.y_hist = np.array([initial_voltage, initial_voltage])

    def measure(self, true_gap: float) -> float:
        voltage = (self.p.sensitivity*true_gap + self.p.bias_voltage
                   + self.rng.normal(0.0, self.p.noise_std_voltage))
        filtered = (self.b[0]*voltage + self.b[1]*self.x_hist[0] + self.b[2]*self.x_hist[1]
                    - self.a[1]*self.y_hist[0] - self.a[2]*self.y_hist[1])
        self.x_hist[1], self.x_hist[0] = self.x_hist[0], voltage
        self.y_hist[1], self.y_hist[0] = self.y_hist[0], filtered
        return filtered/self.p.sensitivity


class NeuralAdaptivePIDController:
    #"""One-hidden-layer NN that adapts bounded PID gains online.

   # NN input z = [e/e_s, de/dt / d_s, integral/e_i, previous_delta_i/i_s].
    #Hidden activation is tanh. Output activation is sigmoid, followed by an
   # affine map into strict positive gain intervals.

   # Delayed gradient derivation
    #---------------------------
   # J_k = 0.5*(e_k/e_s)^2 and e = y-r. Because position cannot respond
   # instantaneously to current, e_k updates the network cache that produced
   # the preceding command. With G = dy/du estimated from the local AMB force
   # gradient and a short adaptation horizon,

   #   dJ/dw = (e_k/e_s^2) * G * du/dK * dK/do * do/dw.

    #Here du/dK=[e, integral(e), filtered_derivative(e)], dK/do contains the
   # sigmoid derivative and gain range, and ordinary backpropagation supplies
   # do/dw. The analytical local estimate is

    #  G ~= -0.5*T_J^2/m * dF/di,
    #  dF/di ~= 2*k_f*i/gap^2.

   # The negative sign follows the gap convention: increasing current reduces
   # gap. Gradient clipping, deadband, bounded gains, weight decay and actuator
   # anti-windup protect the online update.
   # """

    def __init__(self, cp: ControllerParameters, pp: PlantParameters, seed: int = 5):
        self.p, self.plant_p = cp, pp
        rng = np.random.default_rng(seed)
        nh, ni = cp.hidden_neurons, 4
        self.W1 = rng.normal(0.0, 0.12, (nh, ni))
        self.b1 = np.zeros(nh)
        self.W2 = rng.normal(0.0, 0.015, (3, nh))
        lo, hi, initial = map(np.asarray, (cp.gain_min, cp.gain_max, cp.gain_initial))
        target = np.clip((initial-lo)/(hi-lo), 1e-5, 1-1e-5)
        self.b2 = np.log(target/(1-target))
        self.integral = 0.0
        self.derivative = 0.0
        self.previous_error = 0.0
        self.first_sample = True
        self.previous_delta_current = 0.0
        self.previous_cache: Optional[Dict[str, np.ndarray]] = None

    @staticmethod
    def _sigmoid(x: np.ndarray) -> np.ndarray:
        x = np.clip(x, -35.0, 35.0)
        return 1.0/(1.0+np.exp(-x))

    def _gain_forward(self, nn_input: np.ndarray) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
        hidden = np.tanh(self.W1@nn_input+self.b1)
        sigmoid_output = self._sigmoid(self.W2@hidden+self.b2)
        lower, upper = np.asarray(self.p.gain_min), np.asarray(self.p.gain_max)
        gains = lower+(upper-lower)*sigmoid_output
        return gains, {'z': nn_input, 'h': hidden, 's': sigmoid_output}

    def _plant_jacobian(self, gap: float, current: float) -> float:
        p = self.plant_p
        gap = max(gap, p.hard_stop_min)
        dforce_di = 2.0*p.force_coefficient*max(current, 0.05)/gap**2
        actuator_dc = p.amplifier_efficiency*p.amplifier_gain
        return -0.5*self.p.jacobian_horizon**2*dforce_di*actuator_dc/p.mass

    def _adapt(self, observed_error: float, gap: float, current: float) -> None:
        if self.previous_cache is None or abs(observed_error) < self.p.adaptation_deadband:
            return
        c = self.previous_cache
        error_scale = self.p.normalized_error_scale
        dJ_du = (observed_error/error_scale**2)*self._plant_jacobian(gap, current)
        # u = I0 + Kp*e + Ki*integral + Kd*de/dt.
        d_u_d_gain = c['pid_basis']
        gain_range = np.asarray(self.p.gain_max)-np.asarray(self.p.gain_min)
        d_gain_d_output = gain_range*c['s']*(1.0-c['s'])
        delta_out = dJ_du*d_u_d_gain*d_gain_d_output
        grad_W2 = np.outer(delta_out, c['h'])
        grad_b2 = delta_out
        delta_hidden = (self.W2.T@delta_out)*(1.0-c['h']**2)
        grad_W1 = np.outer(delta_hidden, c['z'])
        grad_b1 = delta_hidden

        # Global-norm clipping avoids a single disturbance producing an
        # unsafe parameter jump; weight decay prevents slow parameter drift.
        norm = np.sqrt(np.sum(grad_W1**2)+np.sum(grad_W2**2)
                       +np.sum(grad_b1**2)+np.sum(grad_b2**2))
        scale = min(1.0, self.p.gradient_clip/max(norm, 1e-12))
        eta, decay = self.p.learning_rate, self.p.weight_decay
        self.W1 -= eta*(scale*grad_W1+decay*self.W1)
        self.W2 -= eta*(scale*grad_W2+decay*self.W2)
        self.b1 -= eta*scale*grad_b1
        self.b2 -= eta*scale*grad_b2

    def compute(self, reference: float, measured_gap: float,
                actual_current: float) -> Tuple[float, np.ndarray, float]:
        dt = self.p.sample_time
        error = measured_gap-reference
        # Bumpless start: no artificial derivative impulse at k=0.
        raw_derivative = 0.0 if self.first_sample else (error-self.previous_error)/dt
        alpha = dt/(self.p.derivative_filter_tau+dt)
        self.derivative += alpha*(raw_derivative-self.derivative)
        self._adapt(error, measured_gap, actual_current)

        z = np.array([
            error/self.p.normalized_error_scale,
            self.derivative/0.20,
            self.integral/0.010,
            self.previous_delta_current/1.5,
        ])
        z = np.clip(z, -5.0, 5.0)
        gains, cache = self._gain_forward(z)
        pid_basis = np.array([error, self.integral, self.derivative])
        delta_current_unsat = float(gains@pid_basis)
        command_unsat = self.plant_p.current_bias+delta_current_unsat
        command = float(np.clip(command_unsat, self.plant_p.current_min, self.plant_p.current_max))

        # Back-calculation anti-windup in error-integral units.
        ki = max(gains[1], 1e-6)
        antiwindup = self.p.antiwindup_gain*(command-command_unsat)/ki
        self.integral += dt*(error+antiwindup)
        self.integral = float(np.clip(self.integral, -self.p.integrator_limit,
                                     self.p.integrator_limit))
        cache['pid_basis'] = pid_basis
        self.previous_cache = cache
        self.previous_error = error
        self.first_sample = False
        self.previous_delta_current = command-self.plant_p.current_bias
        return command, gains, error


class SimulationEnvironment:
    #"""Coordinates digital control, delay, sensing and continuous RK4 plant."""

    def __init__(self, plant: ThrustAMBPlant, sensor: SensorFilter,
                 controller: NeuralAdaptivePIDController, duration: float = 3.0):
        self.plant, self.sensor, self.controller = plant, sensor, controller
        self.duration = duration
        self.ts = controller.p.sample_time
        n_delay = max(0, controller.p.computational_delay_samples)
        self.delay_line = [plant.p.current_bias]*(n_delay+1)

    @staticmethod
    def default_load(t: float) -> float:
        #"""Downward load: required 0.5-s step plus later surge and pulse."""
        step = 0.85 if t >= 0.5 else 0.0
        surge = 0.22*np.sin(2*np.pi*1.25*(t-1.35)) if t >= 1.35 else 0.0
        pulse = -0.55 if 2.15 <= t < 2.25 else 0.0
        return step+surge+pulse

    def run(self, load_function: Optional[Callable[[float], float]] = None) -> Dict[str, np.ndarray]:
        load_function = load_function or self.default_load
        n = int(round(self.duration/self.ts))+1
        names = ('time','gap','measured_gap','velocity','current','current_command',
                 'voltage_demand','magnetic_force','load','kp','ki','kd','error')
        log = {name: np.zeros(n) for name in names}
        reference = self.plant.p.nominal_gap

        for k in range(n):
            t = k*self.ts
            x, v, current, _ = self.plant.state
            measured_gap = self.sensor.measure(x)
            command, gains, error = self.controller.compute(
                reference, measured_gap, current)
            delayed_command = self.delay_line.pop(0)
            self.delay_line.append(command)
            load = load_function(t)

            values = (t,x,measured_gap,v,current,command,self.plant.last_voltage_demand,
                      self.plant.last_force,load,*gains,error)
            for name, value in zip(names, values):
                log[name][k] = value
            if k < n-1:
                self.plant.step(delayed_command, load, self.ts)
        return log


def convergence_time(time: np.ndarray, gap: np.ndarray, reference: float,
                     event_time: float = 0.5, tolerance: float = 0.20e-3,
                     dwell: float = 0.12, search_end: float = 1.34) -> float:
    #"""First post-step time error stays inside tolerance for a dwell period."""
    dt = time[1]-time[0]
    window = max(1, int(round(dwell/dt)))
    first = int(np.searchsorted(time,event_time))
    last = min(int(np.searchsorted(time,search_end)),len(time)-window)
    ok = np.abs(gap-reference) <= tolerance
    for k in range(first,last):
        if np.all(ok[k:k+window]):
            return float(time[k]-event_time)
    return float('nan')


def compute_metrics(log: Dict[str, np.ndarray], p: PlantParameters) -> Dict[str, float]:
    error = log['gap']-p.nominal_gap
    settled = convergence_time(log['time'],log['gap'],p.nominal_gap)
    return {
        'position_RMSE_mm': float(1e3*np.sqrt(np.mean(error**2))),
        'peak_gap_deviation_mm': float(1e3*np.max(np.abs(error))),
        'maximum_current_A': float(np.max(log['current'])),
        'maximum_voltage_demand_V': float(np.max(np.abs(log['voltage_demand']))),
        'convergence_time_after_step_s': settled,
        'minimum_gap_mm': float(1e3*np.min(log['gap'])),
        'maximum_gap_mm': float(1e3*np.max(log['gap'])),
        'safety_bound_violations': int(np.sum((log['gap']<p.gap_min)|(log['gap']>p.gap_max))),
    }


def plot_results(log: Dict[str, np.ndarray], pp: PlantParameters,
                 cp: ControllerParameters, output: Path) -> None:
    t = log['time']
    fig, axes = plt.subplots(4,1,figsize=(11,10),sharex=True)
    axes[0].plot(t,1e3*log['gap'],label='true gap',lw=1.6)
    axes[0].plot(t,1e3*log['measured_gap'],label='filtered sensor',alpha=.65,lw=.9)
    axes[0].axhline(1e3*pp.nominal_gap,color='k',ls=':',label='reference')
    axes[0].axhspan(1e3*pp.gap_min,1e3*pp.gap_max,color='#2ca02c',alpha=.08,label='safe gap')
    axes[0].set_ylabel('Air gap [mm]'); axes[0].legend(ncol=4,fontsize=8); axes[0].grid(alpha=.25)

    axes[1].plot(t,log['current'],label='coil current',lw=1.5)
    axes[1].plot(t,log['current_command'],'--',label='command',alpha=.75)
    axv = axes[1].twinx(); axv.plot(t,log['voltage_demand'],color='#d62728',alpha=.6,label='voltage demand')
    axv.axhline(pp.supply_voltage,color='#d62728',ls=':'); axv.axhline(-pp.supply_voltage,color='#d62728',ls=':')
    axes[1].set_ylabel('Current [A]'); axv.set_ylabel('Voltage [V]',color='#d62728')
    axes[1].legend(loc='upper left',fontsize=8); axv.legend(loc='upper right',fontsize=8); axes[1].grid(alpha=.25)

    axes[2].plot(t,log['kp'],label=r'$K_p$'); axes[2].plot(t,log['ki'],label=r'$K_i$')
    axes[2].plot(t,log['kd'],label=r'$K_d$'); axes[2].set_ylabel('Adaptive gains')
    axes[2].legend(ncol=3); axes[2].grid(alpha=.25)

    axes[3].plot(t,log['load'],label='external axial thrust',color='#9467bd')
    axes[3].plot(t,log['magnetic_force'],label='magnetic force',color='#8c564b',alpha=.8)
    axes[3].set_ylabel('Force [N]'); axes[3].set_xlabel('Time [s]')
    axes[3].legend(); axes[3].grid(alpha=.25)
    fig.suptitle('Neural Adaptive PID Control of a Nonlinear Thrust AMB')
    fig.tight_layout(); fig.savefig(output,dpi=180); plt.close(fig)


def main(argv=None) -> Dict[str, float]:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--output-dir',default='neural_adaptive_pid_results')
    parser.add_argument('--duration',type=float,default=3.0)
    parser.add_argument('--seed',type=int,default=12)
    args, _ = parser.parse_known_args(argv)  # ignores Jupyter's -f kernel.json
    output = Path(args.output_dir); output.mkdir(parents=True,exist_ok=True)

    pp, sp, cp = PlantParameters(), SensorParameters(), ControllerParameters()
    sp.sample_time = cp.sample_time
    plant = ThrustAMBPlant(pp)
    sensor = SensorFilter(sp,pp.initial_gap,seed=args.seed)
    controller = NeuralAdaptivePIDController(cp,pp,seed=args.seed+1)
    log = SimulationEnvironment(plant,sensor,controller,args.duration).run()
    metrics = compute_metrics(log,pp)
    plot_results(log,pp,cp,output/'neural_adaptive_pid_amb.png')
    np.savez_compressed(output/'simulation_data.npz',**log)
    with open(output/'metrics.json','w',encoding='utf-8') as f:
        json.dump(metrics,f,indent=2)
    with open(output/'parameters.json','w',encoding='utf-8') as f:
        json.dump({'plant':asdict(pp),'sensor':asdict(sp),'controller':asdict(cp)},f,indent=2)

    print('\nNeural Adaptive PID Thrust AMB validation')
    print('-'*48)
    for key,value in metrics.items():
        if isinstance(value,float) and np.isnan(value):
            shown = 'not reached in validation window'
        elif isinstance(value,float):
            shown = f'{value:.6g}'
        else:
            shown = str(value)
        print(f'{key:34s}: {shown}')
    print(f'Bias/equilibrium current [A]       : {pp.current_bias:.6g}')
    print(f'Results saved to                  : {output.resolve()}')
    return metrics


if __name__ == '__main__':
    main()



Neural Adaptive PID Thrust AMB validation
------------------------------------------------
position_RMSE_mm                  : 0.320349
peak_gap_deviation_mm             : 1.3534
maximum_current_A                 : 2.78911
maximum_voltage_demand_V          : 10.146
convergence_time_after_step_s     : 0.628
minimum_gap_mm                    : 13.3307
maximum_gap_mm                    : 15.3534
safety_bound_violations           : 0
Bias/equilibrium current [A]       : 2.11812
Results saved to                  : C:\Users\User\neural_adaptive_pid_results
